# 06 — Local Qdrant, embeddings, and payload filters

## Scenario: migrate the Acme support corpus to vector search

Northstar Cloud needs semantic retrieval for support questions without sacrificing source provenance or tenant isolation. This notebook first validates the collection/payload contract offline, then shows an optional Docker + Qdrant path. No network service or model download is needed to complete the core lesson.

## 1. Treat collection design as an API contract

A point contains a vector and payload. The payload carries trusted provenance and authorization metadata; it is not a substitute for application policy. Version embedding model, vector size, distance, chunking, corpus revision, and payload schema together. The helpers fail closed if a document lacks source, tenant, chunk ID, or source version.

In [ ]:
from examples.intermediate.qdrant_local import collection_contract, payload_filter, validate_document

contract = collection_contract(384)
document = {
    'id': 'checkout-17#0',
    'text': 'Correlate European checkout latency with the 08:42 deployment before proposing rollback.',
    'source': 'runbooks/checkout.md',
    'metadata': {
        'tenant_id': 'acme', 'tags': ['support'], 'chunk_id': 'checkout-17#0',
        'source_version': '2026.8', 'effective_until': '2026-12-31',
    },
}
validated = validate_document(document)
print(contract)
print(validated)
assert validated['payload']['tenant_id'] == 'acme'

## 2. Create server-side filters from verified identity

The query or model must not choose its own tenant. Derive the filter from authenticated caller claims and apply it in the database query. Apply the same scope to sparse search, reranking, caches, traces, and citations. Filtering after candidate retrieval is too late because identifiers, text, or scores may already have leaked.

In [ ]:
acme_filter = payload_filter('acme', required_tags={'support'})
globex_filter = payload_filter('globex', required_tags={'support'})
print('Acme filter:', acme_filter)
print('Globex filter:', globex_filter)
assert acme_filter != globex_filter
assert acme_filter['must'][0]['match']['value'] == 'acme'

## 3. Optional local Qdrant execution

Install `pip install -e '.[qdrant]'`, run `docker compose up -d qdrant`, then visit `http://localhost:6333/dashboard`. Local Qdrant defaults to no authentication/encryption, so keep it private to your development machine. The adapter uses `query_points` with a required tenant filter and returns payload for citation rendering.

```python
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from examples.intermediate.qdrant_local import index_documents, search

client = QdrantClient(url='http://localhost:6333')
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
index_documents(client, 'acme_support_v1', [document], encoder, encoder.get_sentence_embedding_dimension())
hits = search(client, 'acme_support_v1', 'checkout is slow in Europe', encoder, tenant_id='acme')
```

Use a persistent volume only for non-sensitive local fixtures. Production needs TLS, auth, network segmentation, backup/snapshot strategy, access controls, and an embedding migration plan.

## 4. Compare retrieval systems rather than assuming vectors win

Dense retrieval can improve paraphrase matching. Lexical retrieval commonly remains important for error codes, product IDs, rare names, and exact policy phrases. Measure candidate recall and final context quality on a held-out dataset, then add hybrid fusion and a bounded reranker if they improve the relevant slices within latency/cost budgets.

```text
exact identifier -> lexical or hybrid baseline
conversational paraphrase -> dense candidate retrieval
both signals -> fusion -> bounded reranker -> cited context
```

### Exercises

1. Add a near-duplicate Globex document and test tenant isolation at the filter boundary.
2. Add a revocation/staleness field and decide whether it excludes, labels, or archives a point.
3. Index a second embedding model in a new collection; compare recall@K, p95, storage, and rollout behavior.
4. Add source deletion and reindex tests so stale vectors cannot outlive the source.

### References

- [Qdrant local quickstart](https://qdrant.tech/documentation/quickstart/)
- [Qdrant filtering](https://qdrant.tech/documentation/search/filtering/) and [hybrid queries](https://qdrant.tech/documentation/search/hybrid-queries/)
- [Sentence Transformers semantic search](https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html)